In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor


In [2]:

# Download all kinases from API

url = "https://kinepik.org/api/0/kinases/all"

response = requests.get(url)

# Convert API response to DataFrame
kinase_df = pd.DataFrame(response.json())

# Keep only UniProt ID and Gene Symbol
kinase_df = pd.DataFrame({
    "UniprotID": kinase_df["UniprotID"],
    "GeneName": kinase_df["GeneInfo"].apply(lambda x: x["MappedGene"])
})

# Display table
kinase_df.head()

,UniprotID,GeneName
0,P06239,LCK
1,P12931,SRC
2,P06241,FYN
3,P00519,ABL1
4,P24941,CDK2


In [3]:
print("Total Kinases:", len(kinase_df))

Total Kinases: 504


In [4]:
# get phosphosites for all kinases

def get_phosphosites(kinase_id):

    url = ( "https://kinepik.org/api/0/kinases/specific?"
        f"kinase_ids={kinase_id}&phosphosites=sites"
    )

    response = requests.get(url)
    data = response.json()

    if len(data) ==0:
        return []

    return data[0]["PhosphositesOnKinase"]

In [5]:

# Collect phosphosites for all 504 kinases

all_phosphosites = []

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    sites = get_phosphosites(kinase_id)

    all_phosphosites.append({
        "UniprotID": kinase_id,
        "GeneName": gene,
        "TotalPhosphosites": len(sites),
        "Phosphosites": sites
    })

phosphosite_df = pd.DataFrame(all_phosphosites)

phosphosite_df.head(10)

,UniprotID,GeneName,TotalPhosphosites,Phosphosites
0,P06239,LCK,6,"[LCK(Y394), LCK(Y505), LCK(Y192), LCK(S42), LC..."
1,P12931,SRC,17,"[SRC(Y216), SRC(Y338), SRC(Y419), SRC(Y530), S..."
2,P06241,FYN,13,"[FYN(Y39), FYN(Y420), FYN(Y28), FYN(Y30), FYN(..."
3,P00519,ABL1,37,"[ABL1(S446), ABL1(S465), ABL1(Y393), ABL1(Y226..."
4,P24941,CDK2,9,"[CDK2(T160), CDK2(Y168), CDK2(S46), CDK2(T165)..."
5,O14757,CHEK1,16,"[CHEK1(S301), CHEK1(S286), CHEK1(S345), CHEK1(..."
6,Q96GD4,AURKB,8,"[AURKB(T232), AURKB(T16), AURKB(S7), AURKB(S33..."
7,P06493,CDK1,12,"[CDK1(S39), CDK1(T161), CDK1(T222), CDK1(Y15),..."
8,O15530,PDPK1,26,"[PDPK1(T513), PDPK1(S241), PDPK1(S393), PDPK1(..."
9,P07949,RET,16,"[RET(Y809), RET(Y1090), RET(Y826), RET(Y1029),..."


In [6]:
# Save phosphosite counts

phosphosite_df.to_csv(
    "phosphosite_counts.csv",
    index=False
)

print("Saved phosphosite_counts.csv")

Saved phosphosite_counts.csv


In [7]:
# get fc values for one phosphosite

def get_fc(phosphosite):

    url = ( "https://kinepik.org/api/0/perturbation/fc?"
        f"type=target_phosphosite&id={phosphosite}"
        "&cell_line=NTERA2&confidence=1"
    )

    response = requests.get(url)

    return response.json()

In [8]:
# Download FC data for all phosphosites

fc_rows = []

for _, row in phosphosite_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    for site in row["Phosphosites"]:

        try:
            fc_data = get_fc(site)

            for record in fc_data:

                info = record[site]

                fc_rows.append({
                    "UniprotID": kinase_id,
                    "GeneName": gene,
                    "Phosphosite": site,
                    "Perturbation": info["Perturbation"],
                    "CellLine": info["CellLine"],
                    "FC": float(info["FC"])   # <-- FIXED
                })

        except:
            continue

fc_df = pd.DataFrame(fc_rows)

In [9]:
fc_df.to_csv(
    "fc_NTERA2.csv",
    index=False
)

print("Saved fc_NTERA2.csv")

Saved fc_NTERA2.csv


In [10]:
print("Number of FC rows:", len(fc_rows))

Number of FC rows: 46421


In [10]:
print(fc_df.shape)

fc_df.head()

fc_df.columns

fc_df.info()

(46482, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46482 entries, 0 to 46481
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     46482 non-null  object 
 1   GeneName      46482 non-null  object 
 2   Phosphosite   46482 non-null  object 
 3   Perturbation  46482 non-null  object 
 4   CellLine      46482 non-null  object 
 5   FC            46482 non-null  float64
dtypes: float64(1), object(5)
memory usage: 2.1+ MB


In [11]:
ksea_rows = []

In [12]:
# Download KSEA for every kinase

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    # only perturbations that exist for this kinase
    kinase_fc = fc_df[fc_df["UniprotID"] == kinase_id]

    perturbations = kinase_fc["Perturbation"].unique()

    print(f"{gene}: {len(perturbations)} perturbations")

    for pert in perturbations:

        try:

            url = (
                f"https://kinepik.org/api/0/perturbation/KSEA?"
                f"kinase_ids={kinase_id}"
                f"&perturbations={pert}"
                f"&cell_line=NTERA2"
            )

            response = requests.get(url)
            data = response.json()

            z_score = data[0][kinase_id][pert]["z_score"]

            ksea_rows.append({
                "UniprotID": kinase_id,
                "GeneName": gene,
                "Perturbation": pert,
                "CellLine": "NTERA2",
                "KSEA_z_score": z_score
            })

        except:
            continue

LCK: 0 perturbations
SRC: 61 perturbations
FYN: 0 perturbations
ABL1: 61 perturbations
CDK2: 61 perturbations
CHEK1: 61 perturbations
AURKB: 0 perturbations
CDK1: 61 perturbations
PDPK1: 61 perturbations
RET: 61 perturbations
MAP3K7: 61 perturbations
PRKCQ: 61 perturbations
IKBKB: 61 perturbations
JAK2: 61 perturbations
CHEK2: 0 perturbations
ATM: 61 perturbations
TTK: 61 perturbations
PLK1: 61 perturbations
CAMK2A: 0 perturbations
MAPK3: 61 perturbations
MAPK1: 61 perturbations
EGFR: 61 perturbations
PRKCA: 61 perturbations
CSNK2A1: 0 perturbations
GSK3A: 61 perturbations
GSK3B: 61 perturbations
INSR: 0 perturbations
UHMK1: 0 perturbations
AKT2: 0 perturbations
PAK1: 61 perturbations
MAPK14: 0 perturbations
CDK7: 61 perturbations
KSR1: 61 perturbations
PAK3: 0 perturbations
PRKDC: 61 perturbations
ERBB2: 61 perturbations
NTRK1: 0 perturbations
MAPKAPK5: 61 perturbations
VRK1: 0 perturbations
DYRK2: 61 perturbations
HIPK2: 61 perturbations
AURKA: 0 perturbations
DAPK1: 0 perturbations


In [13]:
ksea_df = pd.DataFrame(ksea_rows)

In [14]:
print(ksea_df.shape)
ksea_df.head()

(13841, 5)


,UniprotID,GeneName,Perturbation,CellLine,KSEA_z_score
0,P12931,SRC,AZD6482,NTERA2,1.812098
1,P12931,SRC,CAL101,NTERA2,1.146269
2,P12931,SRC,GDC0941,NTERA2,1.631653
3,P12931,SRC,HS173,NTERA2,1.744843
4,P12931,SRC,PIK294,NTERA2,-0.228499


In [15]:
ksea_df.to_csv("ksea_NTERA2.csv", index=False)